# FM-PCC Colab Workflow

---

## Run Order
1. Mount Drive
2. Clone/Update FM-PCC
3. Install Miniconda + Create `FMPCC` env
4. Install D3IL
5. Install requirements with pinned compatibility
6. Set runtime env variables
7. Optional W&B relogin
8. Verify dependencies
9. Prepare dataset
10. Smoke test
11. Full train
12. Eval + visualization
13. Archive logs


## 1) Mount Google Drive


In [3]:
import os
from google.colab import drive

drive.mount('/content/drive', force_remount=True)

FMPCC_ROOT = '/content/drive/MyDrive/FMPCC'
REPO = f'{FMPCC_ROOT}/FM-PCC'
os.makedirs(FMPCC_ROOT, exist_ok=True)

print('Drive mounted')
print('Repo path:', REPO)


Mounted at /content/drive
Drive mounted
Repo path: /content/drive/MyDrive/FMPCC/FM-PCC


## 2) Clone or Update FM-PCC


In [2]:
%%bash
set -e

ROOT="/content/drive/MyDrive/FMPCC"
REPO="$ROOT/FM-PCC"

mkdir -p "$ROOT"
cd "$ROOT"

if [ ! -d "$REPO/.git" ]; then
  git clone --recurse-submodules https://github.com/ghubliming/FM-PCC.git
else
  cd "$REPO"
  git pull --ff-only
  git submodule update --init --recursive
fi

echo "Repo ready: $REPO"


Already up to date.
Repo ready: /content/drive/MyDrive/FMPCC/FM-PCC


## 3) Install Miniconda and Create Env

Keeps Python pinned to 3.10 for compatibility with project dependencies.


In [4]:
%%bash
set -e

if [ ! -x "/content/miniconda3/bin/conda" ]; then
  wget -q https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh -O /content/miniconda.sh
  bash /content/miniconda.sh -b -p /content/miniconda3 -u
fi

/content/miniconda3/bin/conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main
/content/miniconda3/bin/conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r

if ! /content/miniconda3/bin/conda env list | grep -q "^FMPCC "; then
  /content/miniconda3/bin/conda create -n FMPCC python=3.10 -y -q
fi

/content/miniconda3/envs/FMPCC/bin/python -V
/content/miniconda3/envs/FMPCC/bin/pip --version


PREFIX=/content/miniconda3
Unpacking bootstrapper...
Unpacking payload...

Installing base environment...

Preparing transaction: ...working... done
Executing transaction: ...working... done
installation finished.
    You currently have a PYTHONPATH environment variable set. This may cause
    unexpected behavior when running the Python interpreter in Miniconda3.
    For best results, please verify that your PYTHONPATH only points to
    directories of packages that are compatible with the Python interpreter
    in Miniconda3: /content/miniconda3
accepted Terms of Service for https://repo.anaconda.com/pkgs/main
accepted Terms of Service for https://repo.anaconda.com/pkgs/r
Jupyter detected...
2 channel Terms of Service accepted
Retrieving notices: ...working... done
Channels:
 - defaults
Platform: linux-64
Solving environment: ...working... done

## Package Plan ##

  environment location: /content/miniconda3/envs/FMPCC

  added / updated specs:
    - python=3.10


The following packag

## 4) Install D3IL (Critical Original Logic)

Uses editable installs for both D3IL core and `gym_avoiding_env`.


In [5]:
%%bash
set -e

PIP="/content/miniconda3/envs/FMPCC/bin/pip"
REPO="/content/drive/MyDrive/FMPCC/FM-PCC"
D3IL="$REPO/d3il"

if [ ! -d "$D3IL/.git" ]; then
  echo "d3il missing/incomplete -> recloning"
  rm -rf "$D3IL"
  git clone https://github.com/ALRhub/d3il.git "$D3IL"
fi

"$PIP" install -e "$D3IL/environments/d3il"
"$PIP" install -e "$D3IL/environments/d3il/envs/gym_avoiding_env"

echo "D3IL installed"


Obtaining file:///content/drive/MyDrive/FMPCC/FM-PCC/d3il/environments/d3il
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Checking if build backend supports build_editable: started
  Checking if build backend supports build_editable: finished with status 'done'
  Getting requirements to build editable: started
  Getting requirements to build editable: finished with status 'done'
  Preparing editable metadata (pyproject.toml): started
  Preparing editable metadata (pyproject.toml): finished with status 'done'
  Building editable for environments.d3il.d3il_sim (pyproject.toml): started
  Building editable for environments.d3il.d3il_sim (pyproject.toml): finished with status 'done'
  Created wheel for environments.d3il.d3il_sim: filename=environments_d3il_d3il_sim-0.2-0.editable-py3-none-any.whl size=2970 sha256=90495ee088ebfbbd83b51b08ecce6136530ce7404c59636fa57c2ab94f9e8349
  Stored in directory: /tmp/pip-ephem-wheel-cache-eozepy

## 5) Install Requirements (Stable, Pinned)


In [10]:
%%bash
set -e

REPO="/content/drive/MyDrive/FMPCC/FM-PCC"
PIP="/content/miniconda3/envs/FMPCC/bin/pip"
PY="/content/miniconda3/envs/FMPCC/bin/python"

cd "$REPO"

# Install project-locked deps first.
"$PIP" install -r requirements.txt

# Quick sanity check
"$PY" - <<'PY'
import numpy, torch
print("numpy:", numpy.__version__)
print("torch:", torch.__version__)
print("cuda:", torch.cuda.is_available())
print("python:", __import__("sys").executable)
PY

  Using cached absl_py-2.1.0-py3-none-any.whl.metadata (2.3 kB)
  Using cached certifi-2024.6.2-py3-none-any.whl.metadata (2.2 kB)
  Using cached charset_normalizer-3.3.2-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (33 kB)
  Using cached click-8.1.7-py3-none-any.whl.metadata (3.0 kB)
  Using cached cloudpickle-3.0.0-py3-none-any.whl.metadata (7.0 kB)
  Using cached glfw-2.7.0-py2.py27.py3.py30.py31.py32.py33.py34.py35.py36.py37.py38-none-manylinux2014_x86_64.whl.metadata (5.4 kB)
  Using cached google_api_core-2.19.0-py3-none-any.whl.metadata (2.7 kB)
  Using cached google_auth-2.29.0-py2.py3-none-any.whl.metadata (4.7 kB)
  Using cached google_cloud_core-2.4.1-py2.py3-none-any.whl.metadata (2.7 kB)
  Using cached google_cloud_storage-2.16.0-py2.py3-none-any.whl.metadata (6.1 kB)
  Using cached google_crc32c-1.5.0-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (2.3 kB)
  Using cached google_resumable_media-2.7.0-py2.py3-none-any.whl.metadata

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cryptography 46.0.5 requires typing-extensions>=4.13.2; python_full_version < "3.11", but you have typing-extensions 4.12.1 which is incompatible.


## 6) Runtime Environment Variables

Includes W&B malformed service cleanup and Colab rendering settings.


In [7]:
import os

FMPCC = '/content/drive/MyDrive/FMPCC/FM-PCC'
D3IL_ROOT = f'{FMPCC}/d3il'
GYM_AV = f'{D3IL_ROOT}/environments/d3il/envs/gym_avoiding_env'

# Append to existing PYTHONPATH instead of overwriting it.
existing_pp = os.environ.get('PYTHONPATH', '')
parts = [FMPCC, D3IL_ROOT, GYM_AV]
if existing_pp:
    parts.append(existing_pp)

os.environ['FMPCC'] = FMPCC
os.environ['PYTHONPATH'] = ':'.join(parts)
os.environ['MUJOCO_GL'] = 'egl'
os.environ['PYOPENGL_PLATFORM'] = 'egl'
os.environ['MPLBACKEND'] = 'agg'

# Clear bad tokens that can crash wandb manager init.
for key in ('WANDB_SERVICE', 'WANDB__SERVICE'):
    os.environ.pop(key, None)

os.chdir(FMPCC)
print('cwd:', os.getcwd())
print('PYTHONPATH:', os.environ['PYTHONPATH'])

cwd: /content/drive/MyDrive/FMPCC/FM-PCC
PYTHONPATH: /content/drive/MyDrive/FMPCC/FM-PCC:/content/drive/MyDrive/FMPCC/FM-PCC/d3il:/content/drive/MyDrive/FMPCC/FM-PCC/d3il/environments/d3il/envs/gym_avoiding_env:/env/python


## 7) Optional W&B Login

In [15]:
import os
from pathlib import Path
import wandb

# Put your API key in this Drive file (single line, no quotes):
# /content/drive/MyDrive/FMPCC/.wandb_api_key
KEY_FILE = Path('/content/drive/MyDrive/FMPCC/.wandb_api_key')

if not KEY_FILE.exists():
    raise FileNotFoundError(
        f'Missing W&B key file: {KEY_FILE}. Create it with your API key on one line.'
    )

api_key = KEY_FILE.read_text(encoding='utf-8').strip()
if not api_key:
    raise ValueError(f'W&B key file is empty: {KEY_FILE}')

# Clear malformed service vars that can break wandb startup.
for k in ('WANDB_SERVICE', 'WANDB__SERVICE'):
    os.environ.pop(k, None)

wandb.finish()
os.environ['WANDB_MODE'] = 'online'
os.environ['WANDB_API_KEY'] = api_key

wandb.login(key=api_key, relogin=True)

print('W&B mode:', os.environ.get('WANDB_MODE'))
print('W&B key file:', KEY_FILE)
print('W&B key loaded:', f'***{api_key[-4:]}' if len(api_key) >= 4 else '***')

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: llmmail2021 (llmmail2021-technical-university-of-munich) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


W&B mode: online
W&B key file: /content/drive/MyDrive/FMPCC/.wandb_api_key
W&B key loaded: ***KogM


## 8) Full Verification

Validates import chain with the exact env interpreter used for training.


In [11]:
%%bash
set -e

/content/miniconda3/envs/FMPCC/bin/python - <<'PY'
import importlib
import sys

pkgs = [
    'torch', 'numpy', 'scipy', 'gym', 'gymnasium', 'gymnasium_robotics',
    'minari', 'wandb', 'mujoco', 'diffusers', 'transformers'
]

ok = True
for p in pkgs:
    try:
        m = importlib.import_module(p)
        v = getattr(m, '__version__', 'unknown')
        print(f'{p:20s} {v}')
    except Exception as e:
        ok = False
        print(f'{p:20s} NOT IMPORTABLE ({type(e).__name__}: {e})')

import numpy, torch
print('numpy pinned:', numpy.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('device:', torch.cuda.get_device_name(0))

# Guard against the known Colab breakage.
major = int(numpy.__version__.split('.')[0])
if major >= 2:
    ok = False
    print('ERROR: numpy 2.x detected, expected 1.26.4 for this workflow')

if not ok:
    sys.exit(2)
PY

torch                2.2.2+cu121
numpy                1.26.4
scipy                1.13.1
gym                  0.26.2
gymnasium            0.29.1
gymnasium_robotics   1.2.4
minari               0.4.3
wandb                0.17.5
mujoco               2.3.7
diffusers            0.31.0
transformers         4.41.2
numpy pinned: 1.26.4
cuda available: True
device: Tesla T4


## 9) Dataset Preparation (Avoiding)

### Option A: Use existing zip from old DPCC path
Keeps your original logic. This exits quickly if avoiding data already exists.


In [12]:
%%bash
set -e

REPO="/content/drive/MyDrive/FMPCC/FM-PCC"
AVOIDING_DATA="$REPO/d3il/environments/dataset/data/avoiding/data"
DATA_ZIP="/content/drive/MyDrive/DPCC/dpcc/d3il/environments/dataset/data/dataset.zip"

if [ -d "$AVOIDING_DATA" ] && [ "$(ls -A "$AVOIDING_DATA")" ]; then
  echo "avoiding data already present: $(ls "$AVOIDING_DATA" | wc -l) files"
  exit 0
fi

if [ ! -f "$DATA_ZIP" ]; then
  echo "dataset zip not found: $DATA_ZIP"
  echo "Skip this cell and use Option B below if needed."
  exit 1
fi

TMP="/content/avoiding_tmp"
rm -rf "$TMP"
mkdir -p "$TMP"
unzip -q "$DATA_ZIP" "avoiding/*" -d "$TMP"
mkdir -p "$REPO/d3il/environments/dataset/data/avoiding"
cp -r "$TMP/avoiding/." "$REPO/d3il/environments/dataset/data/avoiding/"
rm -rf "$TMP"

echo "avoiding dataset ready: $(ls "$AVOIDING_DATA" | wc -l) files"


avoiding data already present: 96 files


### Option B: Download full D3IL dataset zip with gdown (only if Option A unavailable)


```
%%bash
set -e

REPO="/content/drive/MyDrive/FMPCC/FM-PCC"
DATA_DIR="$REPO/d3il/environments/dataset/data"
ZIP_FILE="$DATA_DIR/dataset.zip"

if [ -f "$ZIP_FILE" ]; then
  echo "zip already exists: $ZIP_FILE"
  exit 0
fi

/content/miniconda3/envs/FMPCC/bin/pip install gdown -q
/content/miniconda3/envs/FMPCC/bin/python -m gdown \
  "https://drive.google.com/uc?id=1SQhbhzV85zf_ltnQ8Cbge2lsSWInxVa8" \
  -O "$ZIP_FILE"

echo "downloaded zip: $ZIP_FILE"


## 10) Smoke Test Train

Short check before full run.


## 11) Full Train

Real-time streaming via `!python`.


In [ ]:
!/content/miniconda3/envs/FMPCC/bin/python scripts/train.py --seeds 6 --num-seeds 1 --use-wandb --wandb-project FMPCC


[ train ] Seed source: cli --seeds
[ train ] Training seeds: [6]
[ train ] Saved seed manifest to: logs/avoiding-d3il/diffusion/H8_K20_Dmodels.GaussianDiffusion/seeds_config.json
wandb: Currently logged in as: llmmail2021 (llmmail2021-technical-university-of-munich). Use `wandb login --relogin` to force relogin
wandb: wandb version 0.25.1 is available!  To upgrade, please run:
wandb:  $ pip install wandb --upgrade
wandb: Tracking run with wandb version 0.17.5
wandb: Run data is saved locally in /content/drive/MyDrive/FMPCC/FM-PCC/wandb/run-20260320_144727-zeq1sf29
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run avoiding-d3il-seed-6
wandb: ⭐️ View project at https://wandb.ai/llmmail2021-technical-university-of-munich/FMPCC
wandb: 🚀 View run at https://wandb.ai/llmmail2021-technical-university-of-munich/FMPCC/runs/zeq1sf29

[utils/config ] Config: <class 'diffuser.datasets.sequence.SequenceDataset'>
    discount: 0.99
    env: avoiding-d3il
    horizon: 8
    include_r

## 12) Resume Training (Optional)


```
!/content/miniconda3/envs/FMPCC/bin/python scripts/train.py --seeds 6 --use-wandb --auto-resume --wandb-project FMPCC


## 13) Evaluation and Results


Remember to edit the yaml in /config to choose seeds
and
must write_to_file: True

In [ ]:
!/content/miniconda3/envs/FMPCC/bin/python scripts/eval.py
!/content/miniconda3/envs/FMPCC/bin/python scripts/load_results.py


## 14) Visualization


In [ ]:
!/content/miniconda3/envs/FMPCC/bin/python scripts/visualize_data_constraints.py


## 15) Archive Logs to Drive


In [ ]:
%%bash
set -e

REPO="/content/drive/MyDrive/FMPCC/FM-PCC"
STAMP="$(date +%Y%m%d_%H%M%S)"
OUT="/content/drive/MyDrive/FMPCC/runs_snapshot/$STAMP"

mkdir -p "$OUT"
if [ -d "$REPO/logs" ]; then
  cp -r "$REPO/logs" "$OUT/"
fi
if [ -d "$REPO/wandb" ]; then
  cp -r "$REPO/wandb" "$OUT/"
fi

echo "snapshot saved: $OUT"
